# Neyshekar ASR: fresh controlled baselines

The implementation lives in `neyshekar_experiments/`. All runs start from pinned pretrained base models and frozen manifests. Both architectures use the same CTC-independent data eligibility; CTC conditions share one fixed alphabet. Report WER and CER. No previous experiment outputs are used.

In [ ]:
# Resolve shared code when launched from code/ or the repository root.
import sys
from pathlib import Path

_code_candidates = [Path.cwd(), Path.cwd() / "code"]
CODE = next(
    (
        p.resolve()
        for p in _code_candidates
        if (p / "neyshekar_experiments" / "protocol.py").is_file()
    ),
    None,
)
if CODE is None:
    raise RuntimeError("Launch this notebook from the repository root or its code/ directory.")
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
from neyshekar_experiments.protocol import ROOT
# End notebook bootstrap

import pandas as pd
from IPython.display import display
from neyshekar_experiments.manifests import prepare
from neyshekar_experiments.training import experiment_grid, plan, execute
from neyshekar_experiments.reporting import family_scores, corrected_results, metric_figure

# Imports and reports never start training; paths use the shared repository root.

RUN_TRAINING = False
DATA_SEED = 42  # All manifests are frozen with this seed; optimization seeds are separate.

# Create/verify manifests from source data when this notebook is run.
manifest_summary = prepare()

## 1. Freeze the data

Eligibility is applied before duration matching. Both architectures use identical clip IDs. Re-running this cell verifies existing manifests rather than changing them. Hours are approximate within one selected clip; fitted hours are recorded exactly.

In [ ]:
manifests = prepare()
display(manifests[["name", "clips", "hours"]])

## 2. Specify the experiment

Three optimization seeds share the same training subsets. Every model is evaluated on full Neyshekar and Common Voice tests; the text-disjoint result is sliced from the full Neyshekar hypotheses. No test-based checkpoint selection is used.

In [ ]:
runs = experiment_grid("matched", seeds=(42, 43, 44))
display(plan(runs))

## Zero-shot references
Run Whisper small before adaptation, plus the larger Whisper and MMS references. Each uses the shared full-test decoding and WER/CER pipeline.

In [ ]:
from neyshekar_experiments.training import ZERO_SHOT_MODELS, evaluate_zero_shot

RUN_ZERO_SHOT = False
if RUN_ZERO_SHOT:
    for model in ZERO_SHOT_MODELS:
        evaluate_zero_shot(model)

## 3. Explicit training

Enable on a machine with working CUDA. Each run records provenance and completion metadata. CLI: `python -m neyshekar_experiments run matched`.

In [ ]:
if RUN_TRAINING:
    execute(runs)
else:
    print("Training disabled. The plan above is not a completed experiment.")

## 4. Fresh results

An empty table means the experiments have not completed. No historical results are substituted.

In [ ]:
display(family_scores("matched"))

## 5. All completed runs

In [ ]:
display(corrected_results())